# Phase 07A: Mobile Deployment Entry, Artifact Registry & Export-Readiness

Phase 07A **có thể chạy song song với XLKD-04** vì chỉ sử dụng ba Student KD đã
được khóa tại Phase 4 và authorization của Phase 6B. Notebook này không phụ
thuộc checkpoint hoặc metric của nhánh XLKD, không huấn luyện và không mở
held-out Test.

Mục tiêu:

1. xác minh authorization Phase 7 và toàn bộ manifest nguồn;
2. chọn checkpoint KD tốt nhất theo **CIDEr-D Validation** trong từng cặp;
3. xác minh SHA-256, kích thước và tensor payload của ba checkpoint;
4. đăng ký tokenizer, decoder và preprocessing contract cần cho export;
5. khóa protocol benchmark CPU/mobile và chỉ ra blocker trước khi xuất ONNX;
6. cấp quyền Phase 07B đóng gói model factory và export nếu mọi gate đạt.

Tên cặp chuẩn hóa trong output: **T1-S1, T2-S2, T3-S3**.

## Kaggle inputs bắt buộc

1. Phase 06B Multi-Teacher Evidence Synthesis.
2. Phase 04F T1-S1 Sequence-KD three-seed aggregate.
3. Phase 04M T2-S2 Sequence-KD three-seed aggregate.
4. Phase 04S T3-S3 Logit-KD three-seed aggregate.
5. Phase 04D T1-S1 Sequence-KD seed 123.
6. Phase 04L T2-S2 Sequence-KD seed 2026.
7. Phase 04Q T3-S3 Logit-KD seed 42.
8. Phase 02A Shared Student Pipeline.

## Runtime

- Accelerator: **None / CPU**.
- Internet: **Off**.
- Không cần gắn MS COCO ở Phase 07A.
- XLKD-04 có thể tiếp tục chạy trong một Kaggle session khác.


## Cell 1 — Môi trường, authorization và source discovery


In [1]:
import csv, hashlib, json, os, platform, time
from datetime import datetime, timezone
from pathlib import Path

STARTED = time.perf_counter()
INPUT_ROOT = Path(os.environ.get("KLCN_INPUT_ROOT", "/kaggle/input"))
NOTEBOOK_ROOT = INPUT_ROOT / "notebooks" if (INPUT_ROOT / "notebooks").is_dir() else INPUT_ROOT
OUT = Path(os.environ.get(
    "KLCN_OUTPUT_ROOT",
    "/kaggle/working//phase7_mobile_deployment_entry",
))
AUDIT = OUT / "audits"
CONFIG = OUT / "configs"
RESULT = OUT / "results"
REPORT = OUT / "reports"
MANIFEST = OUT / "manifests"
for directory in [OUT, AUDIT, CONFIG, RESULT, REPORT, MANIFEST]:
    directory.mkdir(parents=True, exist_ok=True)

def now():
    return datetime.now(timezone.utc).isoformat()

def read_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)

def write_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    temporary.replace(path)

def sha256(path, block_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            block = handle.read(block_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()

def exactly_one(name, root=NOTEBOOK_ROOT):
    matches = sorted(Path(root).rglob(name))
    assert len(matches) == 1, (
        f"Cần đúng một {name} dưới {root}; tìm thấy {len(matches)}: {matches}"
    )
    return matches[0].resolve()

def resolve_inside(root, relative_path):
    root = Path(root).resolve()
    target = (root / relative_path).resolve()
    assert target == root or root in target.parents, (root, target)
    return target

AUTH_PATH = exactly_one("PHASE7_MOBILE_DEPLOYMENT_BENCHMARK_ENTRY_AUTHORIZED.json")
P6B_PATH = exactly_one("PHASE6_MULTI_TEACHER_CONFIRMATION_COMPLETE.json")
P2_PATH = exactly_one("PHASE2_SHARED_TRAINING_BUNDLE_READY.json")

AGGREGATE_PATHS = {
    "T1-S1": exactly_one("PHASE4_TEACHER_A_S1_SEQUENCE_KD_THREE_SEED_AGGREGATE_COMPLETE.json"),
    "T2-S2": exactly_one("PHASE4_TEACHER_B_S2_SEQUENCE_KD_THREE_SEED_AGGREGATE_COMPLETE.json"),
    "T3-S3": exactly_one("PHASE4_TEACHER_C_S3_LOGIT_KD_THREE_SEED_AGGREGATE_COMPLETE.json"),
}
RUN_PATHS = {
    "T1-S1": exactly_one("PHASE4_TEACHER_A_S1_SEQUENCE_KD_SEED123_FULL_RUN_COMPLETE.json"),
    "T2-S2": exactly_one("PHASE4_TEACHER_B_S2_SEQUENCE_KD_SEED2026_FULL_RUN_COMPLETE.json"),
    "T3-S3": exactly_one("PHASE4_TEACHER_C_S3_LOGIT_KD_SEED42_FULL_RUN_COMPLETE.json"),
}

AUTH = read_json(AUTH_PATH)
P6B = read_json(P6B_PATH)
P2 = read_json(P2_PATH)
AGGREGATES = {pair: read_json(path) for pair, path in AGGREGATE_PATHS.items()}
RUNS = {pair: read_json(path) for pair, path in RUN_PATHS.items()}

assert AUTH["status"] == "authorized"
assert P6B["status"] == "complete"
assert P2["status"] == "ready"
assert AUTH["new_training_authorized"] is False
assert AUTH["held_out_test_opening_authorized"] is False
assert AUTH["test_set_status"] == "sealed"
assert AUTH["test_set_used"] is False
assert all(marker["status"] == "complete" for marker in AGGREGATES.values())
assert all(marker["status"] == "complete" for marker in RUNS.values())
assert all(marker["test_set_status"] == "sealed" for marker in AGGREGATES.values())
assert all(marker["test_set_used"] is False for marker in AGGREGATES.values())
assert all(marker["test_set_status"] == "sealed" for marker in RUNS.values())
assert all(marker["test_set_used"] is False for marker in RUNS.values())

entry_audit = {
    "schema_version": "phase7a_cell01_entry_v1",
    "created_at_utc": now(),
    "status": "passed",
    "runtime": {
        "python": platform.python_version(),
        "platform": platform.platform(),
        "input_root": str(INPUT_ROOT),
        "notebook_root": str(NOTEBOOK_ROOT),
    },
    "source_markers": {
        "phase7_authorization": str(AUTH_PATH),
        "phase6b_complete": str(P6B_PATH),
        "phase2_shared_bundle": str(P2_PATH),
        "aggregates": {pair: str(path) for pair, path in AGGREGATE_PATHS.items()},
        "selected_runs": {pair: str(path) for pair, path in RUN_PATHS.items()},
    },
    "parallel_with_xlkd04": True,
    "xlkd04_artifact_read": False,
    "new_training_started": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
}
write_json(AUDIT / "phase7a_cell01_entry.json", entry_audit)

print("PHASE 07A CELL 1 — ENTRY")
print("Status                  : PASSED")
print("Parallel with XLKD-04   : YES")
print("XLKD-04 artifact read   : NO")
print("New training            : NOT AUTHORIZED / NOT STARTED")
print("Held-out Test           : SEALED")
print("Progress                : 1 / 5 code cells")


PHASE 07A CELL 1 — ENTRY
Status                  : PASSED
Parallel with XLKD-04   : YES
XLKD-04 artifact read   : NO
New training            : NOT AUTHORIZED / NOT STARTED
Held-out Test           : SEALED
Progress                : 1 / 5 code cells


## Cell 2 — Manifest verification và lựa chọn checkpoint theo Validation


In [2]:
STARTED = time.perf_counter()

def manifest_reference(marker):
    if "manifest" in marker:
        return marker["manifest"]["csv"]
    if "manifests" in marker:
        return marker["manifests"]["csv"]
    raise KeyError("Marker không có manifest/manifests")

def verify_manifest(marker_path, marker):
    reference = manifest_reference(marker)
    manifest_path = resolve_inside(marker_path.parent, reference["relative_path"])
    assert manifest_path.is_file(), manifest_path
    assert sha256(manifest_path) == reference["sha256"]
    with manifest_path.open("r", encoding="utf-8", newline="") as handle:
        rows = list(csv.DictReader(handle))
    expected_rows = marker.get(
        "manifest_rows",
        marker.get("artifact_count", reference.get("rows")),
    )
    if expected_rows is not None:
        assert len(rows) == int(expected_rows), (manifest_path, len(rows), expected_rows)
    for row in rows:
        target = resolve_inside(marker_path.parent, row["relative_path"])
        assert target.is_file(), target
        if row.get("size_bytes"):
            assert target.stat().st_size == int(row["size_bytes"])
        assert sha256(target) == row["sha256"]
    return rows

def verified_summary(marker_path, marker):
    reference = marker["summary"]
    path = resolve_inside(marker_path.parent, reference["relative_path"])
    assert path.is_file(), path
    assert sha256(path) == reference["sha256"]
    return read_json(path), path

source_manifest_counts = {
    "phase2": len(verify_manifest(P2_PATH, P2)),
    **{
        f"aggregate_{pair}": len(verify_manifest(AGGREGATE_PATHS[pair], AGGREGATES[pair]))
        for pair in AGGREGATES
    },
    **{
        f"selected_run_{pair}": len(verify_manifest(RUN_PATHS[pair], RUNS[pair]))
        for pair in RUNS
    },
}

SUMMARIES = {}
SUMMARY_PATHS = {}
for pair in AGGREGATES:
    SUMMARIES[pair], SUMMARY_PATHS[pair] = verified_summary(
        AGGREGATE_PATHS[pair], AGGREGATES[pair]
    )

expected_selection = {"T1-S1": 123, "T2-S2": 2026, "T3-S3": 42}
selection_rows = []
for pair, expected_seed in expected_selection.items():
    per_seed = SUMMARIES[pair]["per_seed"]
    selected = max(per_seed, key=lambda row: float(row["kd_cider_d"]))
    assert int(selected["seed"]) == expected_seed, (pair, selected, expected_seed)
    run = RUNS[pair]
    assert int(run["seed"]) == expected_seed
    assert abs(
        float(run["best_result"]["metrics"]["CIDEr-D"])
        - float(selected["kd_cider_d"])
    ) < 1e-12
    checkpoint_reference = run["best_checkpoint"]
    checkpoint_path = resolve_inside(
        RUN_PATHS[pair].parent,
        checkpoint_reference["relative_path"],
    )
    assert checkpoint_path.is_file(), checkpoint_path
    checkpoint_sha = sha256(checkpoint_path)
    assert checkpoint_sha == checkpoint_reference["sha256"]
    selection_rows.append({
        "pair": pair,
        "seed": expected_seed,
        "student": {
            "T1-S1": "S1 MobileNetV3-Large",
            "T2-S2": "S2 MobileViT-S",
            "T3-S3": "S3 TinyViT-5M",
        }[pair],
        "kd_method": {
            "T1-S1": "Sequence KD alpha=0.5",
            "T2-S2": "Sequence KD",
            "T3-S3": "Direct Logit KD",
        }[pair],
        "selection_split": "validation",
        "best_epoch": int(selected["best_epoch"]),
        "cider_d": float(selected["kd_cider_d"]),
        "bleu_4": float(selected["kd_bleu_4"]),
        "validation_ce": float(selected["kd_validation_ce"]),
        "checkpoint_path": str(checkpoint_path),
        "checkpoint_file": checkpoint_path.name,
        "checkpoint_size_bytes": checkpoint_path.stat().st_size,
        "checkpoint_sha256": checkpoint_sha,
    })

selection_audit = {
    "schema_version": "phase7a_cell02_selection_integrity_v1",
    "created_at_utc": now(),
    "status": "passed",
    "selection_rule": "maximum CIDEr-D on locked Validation within each pair",
    "source_manifest_rows_verified": source_manifest_counts,
    "selected_checkpoints": selection_rows,
    "selection_split": "validation",
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
}
write_json(AUDIT / "phase7a_cell02_selection_integrity.json", selection_audit)
write_json(RESULT / "phase7a_selected_checkpoint_registry.json", {
    "schema_version": "phase7a_selected_checkpoint_registry_v1",
    "created_at_utc": now(),
    "rows": selection_rows,
})
with (RESULT / "phase7a_selected_checkpoint_registry.csv").open(
    "w", encoding="utf-8", newline=""
) as handle:
    fieldnames = list(selection_rows[0].keys())
    writer = csv.DictWriter(handle, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(selection_rows)

print("PHASE 07A CELL 2 — SELECTION AND INTEGRITY")
print("Status                  : PASSED")
for row in selection_rows:
    print(
        f"{row['pair']:7s}                 : seed {row['seed']} | "
        f"CIDEr-D {row['cider_d']:.6f} | {row['checkpoint_size_bytes']/2**20:.2f} MiB"
    )
print("Selection split         : VALIDATION")
print("Held-out Test           : SEALED")
print("Progress                : 2 / 5 code cells")


PHASE 07A CELL 2 — SELECTION AND INTEGRITY
Status                  : PASSED
T1-S1                   : seed 123 | CIDEr-D 1.111200 | 103.33 MiB
T2-S2                   : seed 2026 | CIDEr-D 1.024771 | 124.88 MiB
T3-S3                   : seed 42 | CIDEr-D 1.097713 | 125.45 MiB
Selection split         : VALIDATION
Held-out Test           : SEALED
Progress                : 2 / 5 code cells


## Cell 3 — Tensor inventory và kiểm tra checkpoint payload


In [3]:
STARTED = time.perf_counter()
import torch

def trusted_load_verified_checkpoint(row):
    path = Path(row["checkpoint_path"])
    assert sha256(path) == row["checkpoint_sha256"]
    try:
        return torch.load(path, map_location="cpu", weights_only=True), "weights_only"
    except Exception as weights_only_error:
        # Chỉ fallback vì file đã được đối chiếu SHA-256 với marker nội bộ đã khóa.
        payload = torch.load(path, map_location="cpu", weights_only=False)
        return payload, f"trusted_pickle_after_sha256:{type(weights_only_error).__name__}"

def extract_model_components(payload):
    """Trích riêng encoder/decoder; không đưa optimizer state vào inventory."""
    assert isinstance(payload, dict), type(payload)
    preferred_keys = [
        "student_encoder_state",
        "student_decoder_state",
        "feature_adapter_state",
        "model_state_dict",
        "state_dict",
        "model",
    ]
    components = {}
    for component_key in preferred_keys:
        candidate = payload.get(component_key)
        if not isinstance(candidate, dict):
            continue
        tensors = {
            str(name): value
            for name, value in candidate.items()
            if torch.is_tensor(value)
        }
        if tensors:
            components[component_key] = tensors
    if components:
        return components
    root_tensors = {
        str(name): value
        for name, value in payload.items()
        if torch.is_tensor(value)
    }
    if root_tensors:
        return {"root": root_tensors}
    raise AssertionError(
        "Không tìm thấy student_encoder_state/student_decoder_state hoặc model state_dict"
    )

tensor_rows = []
for row in selection_rows:
    payload, load_mode = trusted_load_verified_checkpoint(row)
    components = extract_model_components(payload)
    tensors = {
        f"{component_name}.{tensor_name}": tensor
        for component_name, component_state in components.items()
        for tensor_name, tensor in component_state.items()
    }
    assert tensors
    non_finite = 0
    for tensor in tensors.values():
        if tensor.is_floating_point():
            non_finite += int((~torch.isfinite(tensor)).sum().item())
    assert non_finite == 0
    state_elements = sum(int(tensor.numel()) for tensor in tensors.values())
    tensor_bytes = sum(int(tensor.numel() * tensor.element_size()) for tensor in tensors.values())
    tensor_rows.append({
        "pair": row["pair"],
        "seed": row["seed"],
        "load_mode": load_mode,
        "model_components": "+".join(components.keys()),
        "model_component_count": len(components),
        "state_tensor_count": len(tensors),
        "state_tensor_elements": state_elements,
        "state_tensor_bytes": tensor_bytes,
        "estimated_fp32_weight_mib": state_elements * 4 / 2**20,
        "estimated_fp16_weight_mib": state_elements * 2 / 2**20,
        "serialized_checkpoint_mib": row["checkpoint_size_bytes"] / 2**20,
        "non_finite_values": non_finite,
        "exact_trainable_parameter_count_status": "pending_model_factory_instantiation_in_phase7b",
    })
    del payload, components, tensors

write_json(RESULT / "phase7a_checkpoint_tensor_inventory.json", {
    "schema_version": "phase7a_checkpoint_tensor_inventory_v1",
    "created_at_utc": now(),
    "important_note": (
        "state_tensor_elements includes parameters and registered buffers; exact trainable "
        "parameter count requires instantiating the locked model factory in Phase 07B"
    ),
    "rows": tensor_rows,
})
with (RESULT / "phase7a_checkpoint_tensor_inventory.csv").open(
    "w", encoding="utf-8", newline=""
) as handle:
    fieldnames = list(tensor_rows[0].keys())
    writer = csv.DictWriter(handle, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(tensor_rows)

write_json(AUDIT / "phase7a_cell03_checkpoint_payload.json", {
    "schema_version": "phase7a_cell03_checkpoint_payload_v1",
    "created_at_utc": now(),
    "status": "passed",
    "rows": tensor_rows,
    "all_checkpoint_hashes_verified_before_load": True,
    "all_state_dicts_present": True,
    "all_tensors_finite": True,
    "new_training_started": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})

print("PHASE 07A CELL 3 — CHECKPOINT PAYLOAD")
print("Status                  : PASSED")
for row in tensor_rows:
    print(
        f"{row['pair']:7s}                 : {row['state_tensor_count']} tensors | "
        f"{row['state_tensor_elements']:,} elements | "
        f"{row['serialized_checkpoint_mib']:.2f} MiB checkpoint"
    )
print("Exact trainable params  : DEFERRED TO PHASE 07B MODEL FACTORY")
print("Progress                : 3 / 5 code cells")


PHASE 07A CELL 3 — CHECKPOINT PAYLOAD
Status                  : PASSED
T1-S1                   : 372 tensors | 9,014,798 elements | 103.33 MiB checkpoint
T2-S2                   : 373 tensors | 10,886,400 elements | 124.88 MiB checkpoint
T3-S3                   : 356 tensors | 10,935,567 elements | 125.45 MiB checkpoint
Exact trainable params  : DEFERRED TO PHASE 07B MODEL FACTORY
Progress                : 3 / 5 code cells


## Cell 4 — Export assets, benchmark contract và blocker registry


In [4]:
STARTED = time.perf_counter()
P2_ROOT = P2_PATH.parent

required_phase2_assets = {
    "shared_decoder_contract": "configs/decoder/shared_transformer_decoder_contract.json",
    "s1_encoder_contract": "configs/encoders/s1_encoder_adapter_contract.json",
    "s2_encoder_contract": "configs/encoders/s2_encoder_adapter_contract.json",
    "s3_encoder_contract": "configs/encoders/s3_encoder_adapter_contract.json",
    "s1_preprocessing": "configs/preprocessing/s1_preprocessing_contract.json",
    "s2_preprocessing": "configs/preprocessing/s2_preprocessing_contract.json",
    "s3_preprocessing": "configs/preprocessing/s3_preprocessing_contract.json",
    "validation_policy": "configs/phase2_validation_and_selection_policy.json",
}
asset_rows = []
for name, relative_path in required_phase2_assets.items():
    path = resolve_inside(P2_ROOT, relative_path)
    assert path.is_file(), path
    asset_rows.append({
        "asset": name,
        "relative_path": relative_path,
        "size_bytes": path.stat().st_size,
        "sha256": sha256(path),
    })

portable_python_sources = sorted(P2_ROOT.rglob("*.py"))
export_readiness = {
    "schema_version": "phase7a_export_readiness_v1",
    "created_at_utc": now(),
    "status": "preflight_passed_with_expected_phase7b_work",
    "ready": {
        "three_selected_checkpoints_sha256_verified": len(selection_rows) == 3,
        "three_checkpoint_state_dicts_readable": len(tensor_rows) == 3,
        "phase2_export_contracts_present": len(asset_rows) == len(required_phase2_assets),
        "held_out_test_remains_sealed": True,
    },
    "phase7b_required_work": [
        "package the locked S1/S2/S3 model factories as importable Python modules",
        "instantiate each model and report exact total/trainable parameter counts",
        "load the selected state_dict with strict=True",
        "export encoder plus autoregressive decoder contract to ONNX opset 17",
        "retain TorchScript fallback when an ONNX operator is unsupported",
        "run deterministic numerical parity before any latency claim",
    ],
    "not_a_failure": (
        "Phase 02A stores architecture contracts and checkpoints but does not provide a "
        "portable .py model factory; Phase 07B is explicitly responsible for packaging it"
    ),
    "portable_python_sources_found": [str(path) for path in portable_python_sources],
    "source_assets": asset_rows,
    "xlkd04_dependency": False,
    "new_training_authorized": False,
    "test_set_status": "sealed",
    "test_set_used": False,
}
write_json(RESULT / "phase7a_export_readiness.json", export_readiness)

benchmark_contract = {
    "schema_version": "phase7_mobile_benchmark_protocol_v1",
    "created_at_utc": now(),
    "status": "locked",
    "objective": "resource-quality Pareto comparison of T1-S1, T2-S2 and T3-S3",
    "candidate_checkpoint_rule": "highest CIDEr-D on Validation within each locked pair",
    "primary_runtime": "ONNX Runtime Mobile",
    "fallback_runtime": "TorchScript only when ONNX export/parity gate fails",
    "export": {
        "onnx_opset": 17,
        "dynamic_batch": False,
        "batch_size": 1,
        "image_shape": [1, 3, 224, 224],
        "decoder_max_length": 32,
        "primary_decode": "greedy/beam-1",
        "secondary_quality_parity_decode": "beam-3",
        "precision_order": ["FP32", "FP16 after FP32 parity"],
        "quantization": "not allowed until FP32 export and parity are locked",
    },
    "parity_gate": {
        "data": "deterministic non-Test calibration subset",
        "images": 100,
        "max_abs_logit_error_fp32": 1e-4,
        "token_sequence_exact_match_rate_min": 0.99,
        "caption_metric_drift": "report; no silent tolerance relaxation",
    },
    "latency": {
        "warmup_runs": 20,
        "measured_runs": 100,
        "thread_counts": [1, 4],
        "report_ms": ["median", "p90", "p95", "mean", "sample_std"],
        "throughput": "images_per_second",
        "include_preprocess": True,
        "include_token_decode": True,
        "exclude_file_io": True,
    },
    "memory_and_size": {
        "peak_rss_mib": True,
        "model_file_mib": True,
        "runtime_package_mib": True,
        "cold_start_ms": True,
    },
    "device_reporting": [
        "device/model",
        "OS/API level",
        "CPU ABI and core count",
        "RAM",
        "runtime version",
        "thermal/power mode",
    ],
    "selection_policy": (
        "Do not crown a single winner by CIDEr-D alone; report the Pareto frontier of "
        "quality, latency, peak RAM and model size"
    ),
    "held_out_test_opening_authorized": False,
    "test_set_status": "sealed",
    "test_set_used": False,
}
write_json(CONFIG / "phase7_mobile_benchmark_protocol.json", benchmark_contract)

write_json(AUDIT / "phase7a_cell04_export_contract.json", {
    "schema_version": "phase7a_cell04_export_contract_v1",
    "created_at_utc": now(),
    "status": "passed",
    "required_assets_verified": len(asset_rows),
    "benchmark_protocol_locked": True,
    "phase7b_work_is_packaging_and_export_not_training": True,
    "xlkd04_dependency": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})

print("PHASE 07A CELL 4 — EXPORT READINESS")
print("Status                  : PASSED WITH PHASE 07B PACKAGING WORK")
print("Phase 02A assets        :", len(asset_rows), "/", len(required_phase2_assets), "VERIFIED")
print("Portable model factory : TO BE PACKAGED IN PHASE 07B")
print("Primary mobile runtime : ONNX Runtime Mobile, opset 17")
print("New training            : NO")
print("XLKD-04 dependency      : NO")
print("Held-out Test           : SEALED")
print("Progress                : 4 / 5 code cells")


PHASE 07A CELL 4 — EXPORT READINESS
Status                  : PASSED WITH PHASE 07B PACKAGING WORK
Phase 02A assets        : 8 / 8 VERIFIED
Portable model factory : TO BE PACKAGED IN PHASE 07B
Primary mobile runtime : ONNX Runtime Mobile, opset 17
New training            : NO
XLKD-04 dependency      : NO
Held-out Test           : SEALED
Progress                : 4 / 5 code cells


## Cell 5 — Hoàn tất Phase 07A và cấp quyền Phase 07B


In [5]:
STARTED = time.perf_counter()

phase7b_authorization = {
    "schema_version": "phase7b_portable_model_factory_export_authorized_v1",
    "created_at_utc": now(),
    "status": "authorized",
    "required_next": (
        "Phase 07B package locked S1/S2/S3 model factories, load selected checkpoints "
        "strictly and export FP32 ONNX/TorchScript candidates"
    ),
    "selected_checkpoints": [
        {
            key: row[key]
            for key in [
                "pair", "seed", "student", "kd_method", "cider_d",
                "checkpoint_file", "checkpoint_size_bytes", "checkpoint_sha256"
            ]
        }
        for row in selection_rows
    ],
    "phase7b_constraints": {
        "new_training": False,
        "validation_reselection": False,
        "checkpoint_substitution": False,
        "test_opening": False,
        "quantization": False,
        "strict_state_dict_load": True,
        "fp32_parity_before_optimization": True,
    },
    "parallel_with_xlkd04": True,
    "xlkd04_dependency": False,
    "test_set_status": "sealed",
    "test_set_used": False,
}
write_json(OUT / "PHASE7B_PORTABLE_MODEL_FACTORY_EXPORT_AUTHORIZED.json", phase7b_authorization)

completion = {
    "schema_version": "phase7a_mobile_deployment_entry_complete_v1",
    "created_at_utc": now(),
    "status": "complete",
    "decision": "phase7a_passed_phase7b_authorized",
    "selected_pairs": ["T1-S1", "T2-S2", "T3-S3"],
    "selected_seeds": {"T1-S1": 123, "T2-S2": 2026, "T3-S3": 42},
    "selection_endpoint": "maximum CIDEr-D on Validation within each pair",
    "artifact_registry_complete": True,
    "checkpoint_payloads_verified": True,
    "benchmark_protocol_locked": True,
    "portable_model_factory_packaging_pending": True,
    "new_training_performed": False,
    "xlkd04_dependency": False,
    "parallel_with_xlkd04": True,
    "test_set_status": "sealed",
    "test_set_used": False,
}

manifest_candidates = []
for directory in [AUDIT, CONFIG, RESULT, REPORT]:
    for path in sorted(directory.rglob("*")):
        if path.is_file():
            manifest_candidates.append(path)
manifest_rows = []
for path in manifest_candidates:
    manifest_rows.append({
        "relative_path": path.relative_to(OUT).as_posix(),
        "size_bytes": path.stat().st_size,
        "sha256": sha256(path),
    })

manifest_csv = MANIFEST / "phase7a_manifest.csv"
with manifest_csv.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=["relative_path", "size_bytes", "sha256"],
    )
    writer.writeheader()
    writer.writerows(manifest_rows)
manifest_json = MANIFEST / "phase7a_manifest.json"
write_json(manifest_json, {
    "schema_version": "phase7a_manifest_v1",
    "created_at_utc": now(),
    "rows": manifest_rows,
})
sha_file = MANIFEST / "SHA256SUMS"
sha_file.write_text(
    "".join(f"{row['sha256']}  {row['relative_path']}\n" for row in manifest_rows),
    encoding="utf-8",
)

completion["manifest"] = {
    "csv": {
        "relative_path": manifest_csv.relative_to(OUT).as_posix(),
        "sha256": sha256(manifest_csv),
        "rows": len(manifest_rows),
    },
    "json": {
        "relative_path": manifest_json.relative_to(OUT).as_posix(),
        "sha256": sha256(manifest_json),
    },
    "sha256sums": {
        "relative_path": sha_file.relative_to(OUT).as_posix(),
        "sha256": sha256(sha_file),
    },
}
completion["manifest_rows"] = len(manifest_rows)
completion["elapsed_seconds_cell05"] = time.perf_counter() - STARTED
write_json(OUT / "PHASE7A_MOBILE_DEPLOYMENT_ENTRY_COMPLETE.json", completion)

print("PHASE 07A — FINAL")
print("Status                  : COMPLETE")
print("Selected T1-S1          : seed 123")
print("Selected T2-S2          : seed 2026")
print("Selected T3-S3          : seed 42")
print("Checkpoint integrity    : VERIFIED")
print("Benchmark protocol      : LOCKED")
print("New training            : NOT PERFORMED")
print("XLKD-04 dependency      : NONE")
print("Parallel with XLKD-04   : YES")
print("Held-out Test           : SEALED / NOT USED")
print("Phase 07B               : AUTHORIZED")
print("Manifest                :", len(manifest_rows), "/", len(manifest_rows), "VERIFIED")
print("Progress                : 5 / 5 code cells")


PHASE 07A — FINAL
Status                  : COMPLETE
Selected T1-S1          : seed 123
Selected T2-S2          : seed 2026
Selected T3-S3          : seed 42
Checkpoint integrity    : VERIFIED
Benchmark protocol      : LOCKED
New training            : NOT PERFORMED
XLKD-04 dependency      : NONE
Parallel with XLKD-04   : YES
Held-out Test           : SEALED / NOT USED
Phase 07B               : AUTHORIZED
Manifest                : 10 / 10 VERIFIED
Progress                : 5 / 5 code cells
